# NB02 — Segmentation Quality Assurance

**Purpose:** Validate the LVSA U-Net segmentation masks before using them in metric computation.
- Run U-Net on all 26 patients (consensus mask across 13 offsets)
- Visual overlay of myocardium, LV blood pool, RV on each patient's GT image
- Mask statistics: pixel counts, shape validity
- Flag problematic patients (empty/tiny masks)
- **Save `seg_masks_cache.pkl`** — reused by nb03–nb07

**Runs on:** Local CPU — U-Net is fast enough without GPU.

## 0. Install Dependencies

In [1]:
import subprocess, sys

packages = [
    'pydicom',
    'numpy',
    'pandas',
    'matplotlib',
    'Pillow',
    'openpyxl',
    'scipy',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
# Install CPU version of PyTorch to save time/bandwidth
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('All dependencies installed / already present.')

All dependencies installed / already present.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

DATA_ROOT  = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
GT_PATH    = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
CKPT_PATH  = r'E:/CINE-Test1/Unet_LVSA_best.pkl'
CACHE_PATH = 'outputs/seg_masks_cache.pkl'
OUT_DIR    = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Set FORCE_RERUN=True to re-run segmentation even if cache exists
FORCE_RERUN = False
print('Config OK')

Config OK


## 1. Load Data

In [3]:
from utils.dicom_loader import load_all_patients
from utils.gt_loader import load_gt

all_data = load_all_patients(DATA_ROOT, verbose=True)
gt_dict  = load_gt(GT_PATH)
print(f'Loaded {len(all_data)} patients')

  Loaded Patient-BEV: 13 images, step=25 Hz
  Loaded Patient-BGT: 13 images, step=25 Hz
  Loaded Patient-BVE: 12 images, step=27 Hz
  Loaded Patient-CEA: 13 images, step=25 Hz
  Loaded Patient-CEJP: 13 images, step=25 Hz
  Loaded Patient-DEM: 13 images, step=25 Hz
  Loaded Patient-FAJ: 13 images, step=25 Hz
  Loaded Patient-FEC: 13 images, step=25 Hz
  Loaded Patient-GJ: 13 images, step=25 Hz
  Loaded Patient-HMJ: 13 images, step=25 Hz
  Loaded Patient-HRJ: 13 images, step=25 Hz
  Loaded Patient-IAM: 13 images, step=25 Hz
  Loaded Patient-INA: 13 images, step=25 Hz
  Loaded Patient-KP: 7 images, step=50 Hz
  Loaded Patient-MJ: 13 images, step=25 Hz
  Loaded Patient-MS: 13 images, step=25 Hz
  Loaded Patient-MSR: 13 images, step=25 Hz
  Loaded Patient-OAF: 13 images, step=25 Hz
  Loaded Patient-PJ: 13 images, step=25 Hz
  Loaded Patient-RAK: 13 images, step=25 Hz
  Loaded Patient-RK: 13 images, step=25 Hz
  Loaded Patient-SK: 13 images, step=25 Hz
  Loaded Patient-SWJ: 13 images, step=2

## 2. Run Segmentation (with cache)

In [4]:
from utils.seg_wrapper import segment_all_patients

seg_cache = segment_all_patients(
    all_data=all_data,
    ckpt_path=CKPT_PATH,
    cache_path=CACHE_PATH,
    force_rerun=FORCE_RERUN,
    verbose=True,
)
print(f'\nSegmentation complete: {len(seg_cache)} patients in cache')

[seg_wrapper] Loading segmentation cache from outputs\seg_masks_cache.pkl

Segmentation complete: 26 patients in cache


## 3. Mask Statistics Table

In [5]:
rows = []
for pid in sorted(seg_cache):
    seg  = seg_cache[pid]
    H, W = seg.myocardium_mask.shape
    myo_px  = int(seg.myocardium_mask.sum())
    pool_px = int(seg.lv_pool_mask.sum())
    rv_px   = int(seg.rv_mask.sum()) if hasattr(seg, 'rv_mask') else 0
    total   = H * W

    flag = ''
    if myo_px == 0:
        flag = 'EMPTY MYO'
    elif myo_px < 50:
        flag = 'TINY MYO'
    elif pid == 'Patient-KP':
        flag = 'KP (7-img apical)'

    rows.append({
        'Patient': pid,
        'HxW': f'{H}x{W}',
        'Myo_px': myo_px,
        'Pool_px': pool_px,
        'RV_px': rv_px,
        'Myo_%': f'{100*myo_px/total:.2f}',
        'Flag': flag,
    })

seg_stats_df = pd.DataFrame(rows)
print(seg_stats_df.to_string(index=False))
seg_stats_df.to_csv(f'{OUT_DIR}/seg_stats.csv', index=False)

problematic = seg_stats_df[seg_stats_df['Flag'] != '']
print(f'\nProblematic patients ({len(problematic)}):')
print(problematic[['Patient','Myo_px','Flag']].to_string(index=False))

     Patient     HxW  Myo_px  Pool_px  RV_px Myo_%      Flag
 Patient-BEV 256x208     814     1237   1258  1.53          
 Patient-BGT 256x208    1026     1843   1764  1.93          
 Patient-BVE 256x208     809     1235   1263  1.52          
 Patient-CEA 256x230     447      687    772  0.76          
Patient-CEJP 256x208     675     1366   1417  1.27          
 Patient-DEM 256x208     667     1628   1339  1.25          
 Patient-FAJ 256x208     641     1125    895  1.20          
 Patient-FEC 208x256     531     1303   1129  1.00          
  Patient-GJ 256x208     674     1257   1116  1.27          
 Patient-HMJ 208x256     269     1419   1445  0.51          
 Patient-HRJ 256x230     817     1229   1210  1.39          
 Patient-IAM 256x208     563     1290   1057  1.06          
 Patient-INA 208x256     546     1221   1073  1.03          
  Patient-KP 256x208       0        0      0  0.00 EMPTY MYO
  Patient-MJ 256x208     350      921    720  0.66          
  Patient-MS 256x208    

## 4. Visual Overlay — All Patients (GT offset image)

In [6]:
import matplotlib.pyplot as plt
from utils.vis_utils import plot_mask_overlay

n_pats = len(seg_cache)
cols = 6
rows_needed = (n_pats + cols - 1) // cols

fig, axes = plt.subplots(rows_needed, cols, figsize=(cols * 2.8, rows_needed * 2.8))
fig.patch.set_facecolor('#0f0f1a')
axes_flat = np.array(axes).ravel()

for ax_i, pid in enumerate(sorted(seg_cache)):
    seg  = seg_cache[pid]
    data = all_data[pid]
    gt_no = gt_dict[pid]['image_no'] if pid in gt_dict else 1
    img = data['images'][gt_no - 1]

    rv_mask = seg.rv_mask if hasattr(seg, 'rv_mask') else None
    plot_mask_overlay(
        img=img,
        myo_mask=seg.myocardium_mask,
        pool_mask=seg.lv_pool_mask,
        rv_mask=rv_mask,
        ax=axes_flat[ax_i],
        title=pid.replace('Patient-', ''),
    )

for ax in axes_flat[n_pats:]:
    ax.set_visible(False)

fig.suptitle('Segmentation QA — GT image for each patient\nGreen=Myo, Blue=LV Pool, Yellow=RV',
             fontsize=11, color='white', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/seg_qa_overlay_all.png', bbox_inches='tight', dpi=120)
plt.show()
print('Saved seg_qa_overlay_all.png')

Saved seg_qa_overlay_all.png


C:\Users\Hp\AppData\Local\Temp\ipykernel_4532\679978235.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Per-Offset Mask Consistency (sample patient)

In [7]:
import sys
sys.path.insert(0, os.path.abspath('..'))
from freqscout_agent.tools.segment import SegmentationTool

DEMO_PID = 'Patient-BGT'
tool = SegmentationTool(checkpoint=CKPT_PATH)

data = all_data[DEMO_PID]
fig, axes = plt.subplots(2, 7, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

for i, (img, hz) in enumerate(zip(data['images'][:7], data['offsets_hz'][:7])):
    seg_single = tool.run(images=[img], offsets_hz=[hz], patient_id=f'{DEMO_PID}_img{i}', source_slice_idx=0)
    plot_mask_overlay(
        img=img, myo_mask=seg_single.myocardium_mask,
        pool_mask=seg_single.lv_pool_mask,
        ax=axes[0, i], title=f'{hz:+.0f}Hz'
    )

for i, (img, hz) in enumerate(zip(data['images'][7:], data['offsets_hz'][7:])):
    seg_single = tool.run(images=[img], offsets_hz=[hz], patient_id=f'{DEMO_PID}_img{i+7}', source_slice_idx=0)
    plot_mask_overlay(
        img=img, myo_mask=seg_single.myocardium_mask,
        pool_mask=seg_single.lv_pool_mask,
        ax=axes[1, i], title=f'{hz:+.0f}Hz'
    )

fig.suptitle(f'{DEMO_PID} — Per-image segmentation consistency across all 13 offsets',
             fontsize=9, color='white')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/seg_per_offset_{DEMO_PID}.png', bbox_inches='tight', dpi=120)
plt.show()
print(f'Segmentation looks consistent across offsets for {DEMO_PID}')

LV pool not detected for Patient-BGT_img10 – using image centre.


Segmentation looks consistent across offsets for Patient-BGT


C:\Users\Hp\AppData\Local\Temp\ipykernel_4532\840092418.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
print('\nNB02 complete. Cache saved at:', CACHE_PATH)
print('Proceed to nb03_signal_metrics.ipynb')


NB02 complete. Cache saved at: outputs/seg_masks_cache.pkl
Proceed to nb03_signal_metrics.ipynb
